<h1>Exercices sur les décorateur</h1>


<a href="https://github.com/Formation-ML-Pro/1.3-Formation-Python/blob/main/03%20Cours%20avancé/02_Decorateurs.ipynb">Lien cours et corrections</a>

<h2>Setup</h2>

In [52]:
# Standard libraries
import random
import logging
from functools import wraps

<h2>Exercice 1 : Logging</h2>

Créez un décorateur journaliser qui, lorsqu’il est appliqué à une fonction :


1. Affiche un log avant l’exécution de la fonction, indiquant le nom de la fonction et les arguments passés.
2. Exécute la fonction.
3. Affiche un log après l’exécution de la fonction, indiquant le résultat retourné.


Ensuite, appliquez ce décorateur à une fonction multiplier(a, b) qui renvoie le produit de deux nombres entiers.

In [5]:
# Define the logging configuration
logging.basicConfig(level=logging.INFO)

# Define a decorateur to manage logs
def logger(func):
    # Use the @wraps decorator to maintain input function metadata 
    @wraps(func)
    # Define the wrapper 
    def wrapper(*args, **kwargs):
        # Define a log before runing the function
        logging.info(f"Call of the function {func.__name__} with {args}, {kwargs}")

        # Run the function
        result = func(*args, **kwargs)

        # Define a log after having run the function
        logging.info(f"{func.__name__} returned {result}")

        return result
    return wrapper

In [6]:
# Associate the decorator with the function
@logger
def multilpy(a:int, b:int) -> int:
    """ Multiply 2 numbers. """
    return a * b

In [7]:
# Run the function 
print(multilpy(3, 4))

INFO:root:Call of the function multilpy with (3, 4), {}
INFO:root:multilpy returned 12


12


<h2>Exercice 2: Entry check for RMSE computation</h2>

Créez un décorateur validate_numeric_inputs qui vérifie que tous les arguments passés à une fonction RMSE sont des nombres (int ou float).

- Si un argument n’est pas numérique, le décorateur doit lever une TypeError avec un message clair.
- Si tous les arguments sont valides, la fonction doit s’exécuter normalement.
Appliquez ce décorateur à une fonction calculate_rmse(predictions, actuals) qui calcule le RMSE (root mean square error) entre deux listes de nombres.

Testez la fonction avec des arguments corrects et incorrects pour montrer que le décorateur fonctionne correctement.

In [40]:
# Define a decorator 
def validate_numeric_inputs(func):
    """ Check that each argument is numeric """
    # Use the `wraps` decorator to maintain the input function metadata
    @wraps(func)
    # Define the wrapper
    def wrapper(*args, **kwargs):
        #print(args)
        # Iterate through each arg
        for arg in args:
            # IF .. the argument is not a number then raise a error
            if not isinstance(arg, (int, float)):
                raise TypeError(f"Every argument must be numeric. Received: {type(arg)}")

        # Iterate through each key word argument
        for value in kwargs.values():
            if not isinstance(value, (int, float)):
                raise TypeError(f"Every argument must be numeric. Received: {type(arg)}")

        # Run the function and return it 
        return func(*args, **kwargs)
    return wrapper


In [41]:
# Define the function to compute the RMSE and associate it the decorator 
@validate_numeric_inputs
def compute_rmse(predictions, actuals):
    """ Compute RMSE between preditions and real values """

    # Compute the number of args of the list
    n = len(predictions)

    # Compute the RMSE
    squared_errors = [(p - a) ** 2 for p, a in zip(predictions, actuals)]
    return (sum(squared_errors)/n) ** 0.5

Il faut bien noter ici que `args` sera un tuple de la forme `(predictions, actuals)`, contenant ainsi nos deux listes d'entrée, donc `([1, 2, 3], [1.1, 2.2, 2.9])`. Ce qui doit déclencher un message d'erreur, comme la condition `isinstance`n'est pas respectée. En effet, `arg` est donc ici une liste et non une valeur numérique.

In [42]:
# Run the function with values that each respects the `isinstance` condition, 
# but as the arg is a tuple of lists, an error message will be raised
try:
    rmse = compute_rmse([1, 2, 3], [1.1, 2.2, 2.9])
    print(f"RMSE: {rmse:.4f}")
except TypeError as e:
    print(f"Error caught: {e}")

Error caught: Every argument must be numeric. Received: <class 'list'>


Il faut donc modifier notre décorateur pour qu'il aille chercher dans la liste les éléments un par un pour appliquer son check conditionnel sur toutes les valeurs. 


In [43]:
# Define a decorator 
def validate_numeric_inputs(func):
    """ Check that each argument is numeric """
    # Use the `wraps` decorator to maintain the input function metadata
    @wraps(func)
    # Define the wrapper
    def wrapper(*args, **kwargs):
        # Iterate through each arg
        for arg in args:
            # IF .. the argument is not a number then raise a error
            if not all(isinstance(x, (int, float)) for x in arg):
                raise TypeError(f"Every argument must be numeric. Received: {type(arg)}")

        # Iterate through each key word argument
        for value in kwargs.values():
            if not all(isinstance(x, (int, float)) for x in value):
                raise TypeError(f"Every argument must be numeric. Received: {type(arg)}")

        # Run the function and return it 
        return func(*args, **kwargs)
    return wrapper


In [44]:
# Define the function to compute the RMSE and associate it the decorator 
@validate_numeric_inputs
def compute_rmse(predictions, actuals):
    """ Compute RMSE between preditions and real values """

    # Compute the number of args of the list
    n = len(predictions)

    # Compute the RMSE
    squared_errors = [(p - a) ** 2 for p, a in zip(predictions, actuals)]
    return (sum(squared_errors)/n) ** 0.5

In [45]:
try:
    rmse = compute_rmse([1, 2, 3], [1.1, 2.2, 2.9])
    print(f"RMSE: {rmse:.4f}")
except TypeError as e:
    print(f"Error caught: {e}")

RMSE: 0.1414


In [47]:
try:
    rmse = compute_rmse([1, "bob" , 3], [1.1, 2.2, 2.9])
    print(f"RMSE: {rmse:.4f}")
except TypeError as e:
    print(f"Error caught: {e}")

Error caught: Every argument must be numeric. Received: <class 'list'>


In [46]:
# Run the function  with values that won't work 
try:
    rmse = compute_rmse("invalid", [1, 2, 3])
    print(f"RMSE: {rmse:.4f}")
except TypeError as e:
    print(f"Error caught: {e}")

Error caught: Every argument must be numeric. Received: <class 'str'>


<h2>Exercice 3 : Retry behavior</h2>

Créer une fonction qui fournie un nombre aléatoire entre 0 et 10 ainsi qu'un décorateur `retry` qui permet de réessayer l'operation un certain nombre de fois, si la valeur obtenue est en dessous d'un certain seuil.

In [53]:
# Define our decorator with inputs
def retry(max_tries: int, threshold: float):
    """ Decorator that retry the function while the value is < threshold """
    # Define the funct that will get the input func
    def input_func(func):
        # Use the `wraps` decorator to maintain the input function metadata
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Initialise counter
            attempt = 1

            # Run the function
            result = func(*args, **kwargs)

            # Run the func until the threshold is reached or the limited number of trial is reached
            while result < threshold and attempt < max_tries:
                print(f"[Retry] Trial {attempt}/{max_tries} : value {result} < threshold {threshold}")
                # Increment counter 
                attempt += 1
                # Run the function
                result = func(*args, **kwargs)

            print(f"[Final] Value obtained: {result}")
            return result

        return wrapper

    return input_func


In [54]:
# Define a function that generat random values, and associate it the decorator 
@retry(max_tries=5, threshold=7)
def random_number():
    return random.randint(0, 10)

In [56]:
# Run the function with the decorator
random_number()

[Retry] Trial 1/5 : value 6 < threshold 7
[Retry] Trial 2/5 : value 5 < threshold 7
[Retry] Trial 3/5 : value 1 < threshold 7
[Retry] Trial 4/5 : value 4 < threshold 7
[Final] Value obtained: 9


9